# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the record sets and their details with @id
record_sets_info = []
print("Record sets available in this dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    record_sets_info.append(rs)
    # List fields in each record set
    print("  Fields:")
    for f in rs.get('fields', []):
        print(f"    - @id: {f['@id']} | Name: {f.get('name', 'N/A')} | dataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record sets by @id, as listed above (update as needed based on printout)
# For this dataset, likely @ids (as an example) are used. Replace with actual output from previous cell if different.
record_set_ids = [rs['@id'] for rs in record_sets_info]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}:", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- Example EDA for one record set. Select numeric and group fields by their @id ---
# You may need to update field names based on cell 6/7 output.

# Example: Choose a record set with numeric data for demonstration.
if dataframes:
    chosen_record_set = list(dataframes.keys())[0]
    df = dataframes[chosen_record_set]
    print(f"\nProceeding with record set: {chosen_record_set}")
    
    # Find numeric columns (fields with float/int types)
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    if numeric_columns:
        numeric_field = numeric_columns[0]
        print(f"Numeric field chosen: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75) # Example: filter above 3rd quartile
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a likely categorical column
        candidate_group_fields = df.select_dtypes(include=['object']).columns.tolist()
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            print(f"\nGrouping data by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print("Grouped data (mean):")
            display(grouped_df.head())
    else:
        print("No numeric fields found in this record set for EDA.")
else:
    print("No DataFrames extracted from the record sets.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization example for numeric field distribution and grouped means
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes and numeric_columns:
    # Distribution plot
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in {chosen_record_set}")
    plt.xlabel(numeric_field)
    plt.show()
    
    # If grouping was performed
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field} (filtered)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated how to load and explore a Croissant-based FAIR dataset using the `mlcroissant` library:

- The metadata provides a rich summary of the dataset structure and provenance.
- Record sets and fields are clearly identified by their `@id`, allowing precise access.
- Data was loaded, overviewed, and explored with basic EDA and normalization steps.
- Plots and grouped summaries provide initial insight for further scientific analysis.

For advanced processing, users can extend these EDA steps, select specific fields by their `@id`, or integrate with machine learning frameworks in a reproducible workflow.